# Generación de predicciones nuevas.


Requisitos:

- Base con propiedad(es) a calificar con el modelo. Se debe contar con los siguientes atributos:
- mts2 : metros cuadrados de la propiedad
- banos: Cantidad de baños que posee la propiedad.
- tipo_vivienda: Tipo de vivienda (casa/apartamento/otros).
- dormitorios: Cantidad de dormitorios que pasee la propiedad.

Los siguientes atributos se obtienen del notebook "00- Obtención de datos generales.ipynb" en esta misma carpeta:
- CANT_HOSPITAL: Cantidad de hospitales cercanos a la propiedad.
- CANT_SUPERMERCADO: Cantidad de supermercados cercanos a la propiedad.
- miembros_ph_avg: Promedio de miembros por hogar del departamento en el que se encuentra la propiedad.
- hogares_lectura: Porcentaje de hogares con alfabetización del depto. en el que se encuentra la propiedad.
- hogares_techo_lamina: Porcentaje de hogares con techo de lámina en el depto. en el que se encuentra la propiedad.
- hogares_sin_agua: Porcentaje de hogares sin acceso a agua potable en el depto. en el que se encuentra la propiedad.



Output:
- Predicción(es) del precio para cada propiedad a analizar.

In [ ]:
import pandas as pd
import numpy as np
import openpyxl

import matplotlib.pyplot as plt



In [ ]:
datos = pd.read_csv('base_atributos_propiedad.csv') #otorgada por usuario; debe contener el(las) propiedad(es) a calificar

datos.info()

In [ ]:
# Archivo obtenido del notebook "00-Obtencion de datos generales"
datos_depto = pd.read_csv('atributos_departamento.csv')
datos_depto.info()

In [ ]:
# Archivo obtenido del notebook "00-Obtencion de datos generales"
datos_poi = pd.read_csv('POIS.csv')
datos_poi.info()

In [ ]:
datos = datos.merge(datos_depto, how = 'left', left_on = 'nom_dpto',right_on = 'DEPTO' )
datos = datos.merge(datos_poi, how = 'left', on = ['latitud','longitud'] )

## Preprocesamiento

Imputación de nulos y tratamiento de outliers

In [ ]:
datos['mts2'] = datos['mts2'].fillna(-9999)
datos['banos'] = datos['banos'].fillna(3.3866666666666667)
datos['dormitorios'] = datos['dormitorios'].fillna(3.6573426573426575)


In [ ]:
datos['mts2'] = datos['mts2'].clip(lower=50.15, upper=1070741.05)
datos['dormitorios'] = datos['dormitorios'].clip(lower=1, upper=10)
datos['banos'] = datos['banos'].clip(lower=1, upper=17)

datos['CANT_HOSPITAL'] = datos['CANT_HOSPITAL'].clip(lower=0, upper=28)
datos['CANT_SUPERMERCADO'] = datos['CANT_SUPERMERCADO'].clip(lower=0, upper=24)


datos['hogares_lectura'] = datos['hogares_lectura'].clip(lower=0.7334933973589436, upper=0.8524151041216624)
datos['hogares_techo_lamina'] = datos['hogares_techo_lamina'].clip(lower=0.2864345738295318, upper=0.801221684907101)
datos['hogares_sin_agua'] = datos['hogares_sin_agua'].clip(lower=0.0948571428571428, upper=0.3375750300120048)
datos['miembros_ph_avg'] = datos['miembros_ph_avg'].clip(lower=3.762132352941177, upper=4.217103588699414)



In [ ]:
datos['tipo_vivienda_avg'] = np.where(datos['tipo_vivienda']=='APARTAMENTO', 309561.158537,
                                      np.where(datos['tipo_vivienda']=='CASA',427654.239353,
                                               994828.335000))

## Predicción

In [ ]:
atributos = ['mts2', 'banos', 'miembros_ph_avg', 'hogares_lectura', 'CANT_HOSPITAL', 'parqueos', 'CANT_SUPERMERCADO', 'tipo_vivienda_avg', 'hogares_techo_lamina', 'hogares_sin_agua', 'dormitorios'
]

In [ ]:
import pickle

with open("modelo_rf.pkl", "rb") as f:
    modelo = pickle.load(f)
    
with open("power_transformer.pkl", "rb") as f:
    pt = pickle.load(f)

In [ ]:
datos['prediccion'] = pt.inverse_transform(modelo.predict(datos[atributos]).reshape(-1, 1)).flatten()

datos['prediccion'].describe()

In [ ]:
# Opcional: Exportar

datos.to_csv('base_propiedades_predicciones.csv', index = False)

## Opcional: Interpretabilidad

Explorar la forma en que el modelo obtuvo las predicciones para propiedades particulares.


In [ ]:
!pip install lime

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
def predict_fn(X):
    # X llega como numpy array
    X_df = pd.DataFrame(X, columns=atributos)
    
    # Predicción en escala transformada
    y_pred_t = modelo.predict(X_df)
    
    # Invertir transformación del target
    y_pred = pt.inverse_transform(y_pred_t.reshape(-1, 1)).flatten()
    
    return y_pred

In [ ]:
explainer = LimeTabularExplainer(
    training_data=datos[atributos].values,
    feature_names=atributos,
    mode="regression",
    discretize_continuous=True
)

In [ ]:
i = 10  # índice de la observación a explicar

exp = explainer.explain_instance(
    datos.iloc[i][atributos].values,
    predict_fn,
    num_features=10
)

In [ ]:
exp.show_in_notebook(show_table=True)

#